# Track 3 / Challenge 1: environment and candidate CSV
This notebook checks the runtime and reads the local candidate contract. It installs no packages or models and generates no designs.

GPU access is optional for this step. In Colab, select **Runtime → Change runtime type → GPU** if desired.

CSV validation and folding scores do not prove binding, mouse cross-reactivity, or pH-selective binding. Consult the repository README for assumptions and the [official challenge](https://proteinbase.com/competitions/anthropic-adaptyv-2026/challenges/egfr) for current rules.


In [ ]:
import platform
import shutil
import subprocess

print("Python:", platform.python_version())
smi = shutil.which("nvidia-smi")
if smi:
    try:
        result = subprocess.run(
            [smi, "--query-gpu=name,memory.total,driver_version", "--format=csv"],
            capture_output=True, text=True, timeout=15,
        )
        print("NVIDIA GPU available:", result.returncode == 0)
        print(result.stdout if result.returncode == 0 else result.stderr)
    except (OSError, subprocess.TimeoutExpired) as exc:
        print("GPU query unavailable:", exc)
else:
    print("NVIDIA GPU available: False (nvidia-smi not found). CPU is sufficient here.")
print("This checks NVIDIA device visibility, not compatibility with any model framework.")


## Locate repository files
Locally, run from the repository root or its `notebooks/` directory. In Colab, upload `data/candidates.csv`, `data/candidates.schema.json`, and `scripts/validate_candidates.py` when prompted. Only upload a validator you trust: a later cell executes it.


In [ ]:
from pathlib import Path

required = ("data/candidates.csv", "data/candidates.schema.json", "scripts/validate_candidates.py")
ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
             if all((p / name).is_file() for name in required)), None)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Run from the repository root or notebooks directory.") from exc
    import tempfile
    ROOT = Path(tempfile.mkdtemp(prefix="adaptyv_"))
    print("Upload candidates.csv, candidates.schema.json, and validate_candidates.py")
    uploaded = files.upload()
    missing = [Path(name).name for name in required if Path(name).name not in uploaded]
    if missing:
        raise ValueError(f"Missing uploads: {missing}; rerun this cell with all three files.")
    for name in required:
        destination = ROOT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(uploaded[Path(name).name])
print("Workspace:", ROOT)


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("candidate_validator", ROOT / "scripts/validate_candidates.py")
validator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validator)

ALLOW_EMPTY = True  # Scaffold only; set False when preparing actual candidates.
CSV_PATH = ROOT / "data/candidates.csv"
rows, errors = validator.validate_csv(CSV_PATH, allow_empty=ALLOW_EMPTY)
if errors:
    raise ValueError("CSV validation failed:\n" + "\n".join(errors))
print(f"Read {len(rows)} candidates from {CSV_PATH}")
if not rows:
    print("Header-only scaffold: no candidate designs present.")
for rank, row in enumerate(rows[:5], start=1):
    print(rank, row)
print("CSV hygiene only; biological eligibility and experimental performance are unverified.")
